<a href="https://colab.research.google.com/github/prat-kap/Assignment-Task-Management/blob/main/Copy_of_Mini_Project_Review_Present.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎤 Mini-Project: Review & Present
### AIM Program | IIT Patna × Masai School

---

## What This Notebook Covers

This notebook teaches you how to **review your own ML project critically** and **present it clearly**.
Work through every section — the skills here are what separate a junior data scientist from one worth hiring.

| Section | Core Skill | What You Produce |
|---------|-----------|------------------|
| 1 — Code Walkthrough | Structure · Clarity · Reproducibility | Annotated checklist |
| 2 — Honest Metrics | Interpret results in business context | Dashboard with commentary |
| 3 — Error Analysis | Find where the model fails and why | Slice analysis + failure patterns |
| 4 — Improvement Suggestions | Evidence-backed, specific next steps | Prioritised roadmap |
| 5 — Demo Rehearsal | 5-minute structured presentation | Script + timing |
| 6 — Peer Feedback | Give and receive structured critique | Structured feedback form |

**Estimated time:** 75–90 minutes

---

> 📌 **The single most important habit from this session:**
> A presentation that honestly acknowledges a model's weaknesses and proposes
> specific, evidence-backed fixes is worth more than one that cherry-picks the
> best numbers and stops there. **Honest interpretation is the skill that gets you hired.**

---
## Section 0 — Setup

Run this cell first. Every section depends on it.

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib, os, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection   import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline          import Pipeline
from sklearn.preprocessing     import StandardScaler, OneHotEncoder
from sklearn.impute            import SimpleImputer
from sklearn.compose           import ColumnTransformer
from sklearn.ensemble          import RandomForestClassifier
from sklearn.linear_model      import LogisticRegression
from sklearn.dummy             import DummyClassifier
from sklearn.metrics           import (
    accuracy_score, f1_score, roc_auc_score, precision_score, recall_score,
    confusion_matrix, classification_report, roc_curve, precision_recall_curve,
    ConfusionMatrixDisplay
)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
SEED = 42
np.random.seed(SEED)
for d in ['data/raw','data/processed','models','outputs']:
    os.makedirs(d, exist_ok=True)

print('✅ Environment ready.')

---
# Section 1 — Code Walkthrough: Structure, Clarity, Reproducibility

> **Core question:** Can someone else — or future-you in 3 months — understand,
> reproduce, and build on this notebook without asking you any questions?

### The Three-Pillar Framework

| Pillar | What it asks | Typical failure |
|--------|-------------|----------------|
| **Structure** | Does the notebook tell a linear story? | Cells run out of order; scratch cells left in |
| **Clarity** | Does every line communicate intent? | `df2`, `m_final`, zero comments |
| **Reproducibility** | Can anyone rebuild your exact results? | No seed; undocumented data source |

In [ ]:
# ── PILLAR 1: STRUCTURE CHECKLIST ────────────────────────────────────────
# Run this checklist on your own notebook before any presentation.

def structure_checklist():
    items = [
        ('Problem statement visible in the first cell',
         'Tells the reader WHAT you are solving and WHY it matters.'),
        ('Data source documented: path, date, row count, description',
         'Anyone must be able to find and reload the exact same data.'),
        ('All cells execute top-to-bottom without errors',
         'Test: Kernel → Restart & Run All. If it fails → not reproducible.'),
        ('No unexplained intermediate DataFrames (df2, df_v3_final)',
         'Each transformation should have a clear, descriptive name.'),
        ('EDA section shows only key insights — exploratory dead ends removed',
         'A presentation is not a lab notebook.'),
        ('Section headers match the actual content below them',
         'A header saying Feature Engineering should contain feature engineering.'),
        ('Summary and next steps present at the end',
         'Reader should know what you found and what you would do next.'),
    ]
    print('📋 NOTEBOOK STRUCTURE CHECKLIST')
    print('='*58)
    for i, (item, why) in enumerate(items, 1):
        print(f'  {i}. [ ] {item}')
        print(f'       → {why}')
        print()

structure_checklist()

In [ ]:
# ── PILLAR 2: CLARITY — Before and After Comparison ─────────────────────

print('VARIABLE NAMING: Before vs After')
print('='*55)
print('''
❌ UNCLEAR  — reviewer must guess what everything means:

  df2   = df.dropna()
  x     = df2.drop('y', axis=1)
  m     = RandomForestClassifier()
  res   = m.score(xt, yt)

✅ CLEAR  — code is self-documenting:

  # Drop the 38 rows where the target label is missing
  df_valid    = df.dropna(subset=['attrition'])

  features    = df_valid.drop(columns=['attrition'])
  target      = df_valid['attrition']

  rf_model    = RandomForestClassifier(
      n_estimators = 200,
      class_weight = "balanced",  # corrects for 20% positive rate
      random_state = SEED
  )
  test_accuracy = rf_model.score(X_test, y_test)
''')

print('FUNCTION DOCUMENTATION: Before vs After')
print('='*55)
print('''
❌ UNCLEAR:
  def proc(d, t, s=42):
      return train_test_split(d.drop(t,1), d[t],
                               test_size=0.2, random_state=s, stratify=d[t])

✅ CLEAR:
  def stratified_split(df, target_col, test_size=0.20, seed=42):
      """
      Split into stratified train/test sets, preserving class balance.
      Parameters: df, target_col, test_size, seed
      Returns: X_train, X_test, y_train, y_test
      """
      X = df.drop(columns=[target_col])
      y = df[target_col]
      return train_test_split(X, y, test_size=test_size,
                               random_state=seed, stratify=y)
''')

In [ ]:
# ── PILLAR 3: REPRODUCIBILITY CHECKLIST ──────────────────────────────────

print('REPRODUCIBILITY — what every shareable notebook must have')
print('='*58)

# 1. Document data source precisely
DATA_SOURCE = {
    'path':        'data/raw/hr_attrition_synthetic.csv',
    'description': 'Synthetic IBM HR Attrition dataset — built in this notebook',
    'rows':        1470,
    'columns':     20,
    'generated':   f'SEED={SEED}',
}
print('\n1. Data source documented:')
for k, v in DATA_SOURCE.items():
    print(f'   {k}: {v}')

# 2. Fixed seed everywhere
print(f'\n2. SEED = {SEED} — used in ALL model and split calls')

# 3. Library versions captured
import sklearn, matplotlib, seaborn
print('\n3. Library versions (requirements.txt):')
reqs = [
    f'scikit-learn=={sklearn.__version__}',
    f'pandas=={pd.__version__}',
    f'numpy=={np.__version__}',
    f'matplotlib=={matplotlib.__version__}',
    f'seaborn=={seaborn.__version__}',
]
for r in reqs:
    print(f'   {r}')
with open('requirements.txt', 'w') as f:
    f.write('\n'.join(reqs))
print('   → written to requirements.txt')

print('\n⚠️  The reproducibility test:')
print('   Kernel → Restart & Run All')
print('   If any cell fails OR produces different output → not reproducible.')
print('   Fix BEFORE sharing your work.')

---
## Section 2 — Build the Project

We build a complete HR attrition prediction project below — intentionally with
some weaknesses that error analysis will reveal. Every cell is annotated as a model
of good practice for your own projects.

In your own project, replace the dataset generation cell with `pd.read_csv('data/raw/...')`.

In [ ]:
# ── GENERATE HR ATTRITION DATASET ────────────────────────────────────────
# Realistic synthetic dataset based on IBM HR Attrition structure.
# Known patterns are baked in so error analysis will find them.

np.random.seed(SEED)
N = 1470

df = pd.DataFrame({
    # Target
    'Attrition':            np.random.choice(['Yes','No'], N, p=[0.20, 0.80]),

    # Numerical features
    'Age':                  np.random.randint(18, 61, N),
    'MonthlyIncome':        np.round(np.random.lognormal(9.5, 0.6, N)).astype(int).clip(1000, 200000),
    'YearsAtCompany':       np.random.randint(0, 41, N),
    'YearsInCurrentRole':   np.random.randint(0, 19, N),
    'TotalWorkingYears':    np.random.randint(0, 41, N),
    'DistanceFromHome':     np.random.randint(1, 30, N),
    'JobSatisfaction':      np.random.choice([1,2,3,4], N, p=[0.08,0.17,0.42,0.33]),
    'WorkLifeBalance':      np.random.choice([1,2,3,4], N, p=[0.05,0.20,0.50,0.25]),
    'PercentSalaryHike':    np.random.randint(11, 26, N),
    'TrainingTimesLastYear':np.random.randint(0, 7, N),
    'JobLevel':             np.random.choice([1,2,3,4,5], N, p=[0.27,0.34,0.18,0.13,0.08]),
    'StockOptionLevel':     np.random.choice([0,1,2,3], N, p=[0.40,0.36,0.14,0.10]),
    'NumCompaniesWorked':   np.random.randint(0, 10, N),

    # Categorical features
    'Department':           np.random.choice(['Sales','R&D','HR'], N, p=[0.30,0.60,0.10]),
    'JobRole':              np.random.choice(['Sales Exec','Research Scientist',
                                              'Lab Tech','Manager','HR Rep'], N),
    'MaritalStatus':        np.random.choice(['Single','Married','Divorced'], N,
                                              p=[0.32,0.46,0.22]),
    'OverTime':             np.random.choice(['Yes','No'], N, p=[0.30, 0.70]),
    'BusinessTravel':       np.random.choice(['Non-Travel','Travel_Rarely',
                                              'Travel_Frequently'], N, p=[0.15,0.70,0.15]),
    'EducationField':       np.random.choice(['Life Sciences','Medical',
                                              'Marketing','Technical Degree','Other'], N),
})

# Introduce realistic missing values (4% of MonthlyIncome)
df.loc[df.sample(frac=0.04, random_state=1).index, 'MonthlyIncome'] = np.nan

# Make OverTime correlate with attrition (realistic signal)
ot_idx = df[df['OverTime']=='Yes'].sample(frac=0.20, random_state=2).index
df.loc[ot_idx, 'Attrition'] = 'Yes'

# Save raw data
df.to_csv('data/raw/hr_attrition_synthetic.csv', index=False)

print(f'Dataset: {df.shape[0]:,} employees × {df.shape[1]} columns')
print(f'Attrition rate: {(df["Attrition"]=="Yes").mean():.1%}')
print(f'Missing values: {df.isnull().sum().sum()} (in MonthlyIncome only)')
display(df.head(3))

In [ ]:
# ── BUILD AND TRAIN ALL THREE MODELS ─────────────────────────────────────
# Baseline → Logistic Regression → Random Forest
# All wrapped in Pipelines (no data leakage)

X = df.drop(columns=['Attrition'])
y = (df['Attrition'] == 'Yes').astype(int)

# Stratified split — mandatory for imbalanced classification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

num_features = X_train.select_dtypes(include='number').columns.tolist()
cat_features = X_train.select_dtypes(include='object').columns.tolist()

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc',  StandardScaler())]),      num_features),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('enc', OneHotEncoder(handle_unknown='ignore',
                                            sparse_output=False))]), cat_features),
])

baseline = Pipeline([('pre', preprocessor),
                     ('mdl', DummyClassifier(strategy='most_frequent', random_state=SEED))])

lr = Pipeline([('pre', preprocessor),
               ('mdl', LogisticRegression(class_weight='balanced',
                                          max_iter=1000, random_state=SEED))])

rf = Pipeline([('pre', preprocessor),
               ('mdl', RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                               random_state=SEED, n_jobs=-1))])

for name, pipe in [('Baseline', baseline), ('Logistic Regression', lr), ('Random Forest', rf)]:
    pipe.fit(X_train, y_train)
    print(f'✅ {name} trained')

y_pred_bl  = baseline.predict(X_test)
y_pred_lr  = lr.predict(X_test)
y_pred_rf  = rf.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:, 1]
y_proba_rf = rf.predict_proba(X_test)[:, 1]

---
# Section 3 — Honest Result Interpretation

Numbers without context are just numbers. The commentary alongside the metric
is as important as the metric itself.

### The 5 Rules of Honest Interpretation

1. **Always compare to a baseline** — show the floor first
2. **Use the metric that matches the business problem** — not the metric that flatters you
3. **Translate scores into business units** — not percentages, but employees and decisions
4. **Show the full confusion matrix** — both types of error matter
5. **Report performance on subgroups** — overall accuracy can mask alarming gaps

In [ ]:
# ── HONEST METRICS FUNCTION ──────────────────────────────────────────────
# Reports every metric WITH a plain-English business interpretation.
# The commentary is deliberate — it forces you to think about meaning.

def honest_metrics(y_true, y_pred, y_proba=None,
                   model_name='Model', baseline_acc=None):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    auc  = roc_auc_score(y_true, y_proba) if y_proba is not None else None

    total    = len(y_true)
    leavers  = int(y_true.sum())
    caught   = int((y_pred * y_true).sum())
    missed   = leavers - caught
    fp       = int((y_pred * (1 - y_true)).sum())

    print(f'\n{"─"*60}')
    print(f'  {model_name}')
    print(f'{"─"*60}')
    print(f'  Accuracy:  {acc:.4f}', end='')
    if baseline_acc:
        delta = acc - baseline_acc
        marker = '▲' if delta > 0 else '▼'
        print(f'   {marker} {abs(delta)*100:.1f} pts vs baseline', end='')
    print()
    print(f'  Precision: {prec:.4f}   → of all flagged employees, {prec*100:.0f}% actually left')
    print(f'  Recall:    {rec:.4f}   → caught {rec*100:.0f}% of actual leavers')
    print(f'  F1-score:  {f1:.4f}   → harmonic mean of precision and recall')
    if auc:
        print(f'  ROC-AUC:   {auc:.4f}   → discrimination across all thresholds')

    print(f'\n  In plain numbers  ({total} test employees, {leavers} actual leavers):')
    print(f'    Correctly flagged: {caught:>4}  ← receive retention intervention')
    print(f'    Missed leavers:    {missed:>4}  ← leave without any intervention')
    print(f'    False alarms:      {fp:>4}  ← non-leavers contacted unnecessarily')

    if rec < 0.55:
        print(f'\n  ⚠️  Recall under 55% — missing {(1-rec)*100:.0f}% of leavers is very high.')
        print(f'     Action: lower threshold, adjust class_weight, or add features.')
    elif rec < 0.75:
        print(f'\n  ⚡ Moderate recall. {missed} missed leavers = lost retention opportunities.')
    else:
        print(f'\n  ✅ Strong recall — catching most at-risk employees.')

    return {'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1, 'auc': auc}

bl_acc = accuracy_score(y_test, y_pred_bl)
m_bl   = honest_metrics(y_test, y_pred_bl,
                         model_name='Baseline — always predict Stay')
m_lr   = honest_metrics(y_test, y_pred_lr, y_proba_lr,
                         model_name='Logistic Regression', baseline_acc=bl_acc)
m_rf   = honest_metrics(y_test, y_pred_rf, y_proba_rf,
                         model_name='Random Forest',       baseline_acc=bl_acc)

In [ ]:
# ── 5-PANEL EVALUATION DASHBOARD ─────────────────────────────────────────
fig = plt.figure(figsize=(20, 10))
gs  = gridspec.GridSpec(2, 5, figure=fig, hspace=0.5, wspace=0.45)
fig.suptitle('Model Evaluation Dashboard — HR Attrition Prediction',
             fontsize=14, fontweight='bold', y=1.01)

# ── Panel 1: Metric bar chart ──────────────────────────────────────
ax1  = fig.add_subplot(gs[0, :2])
mnames  = ['Baseline','Log.Reg.','Rand.Forest']
mvals   = [m_bl, m_lr, m_rf]
mk      = ['acc','prec','rec','f1']
mlabels = ['Accuracy','Precision','Recall','F1']
x = np.arange(len(mk))
width = 0.25
colors= ['#90CAF9','#1565C0','#D32F2F']
for i,(nm,mv) in enumerate(zip(mnames,mvals)):
    vals   = [mv[k] for k in mk]
    offset = (i-1)*width
    bars   = ax1.bar(x+offset, vals, width, label=nm, color=colors[i], alpha=0.9)
    for bar,val in zip(bars,vals):
        ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{val:.2f}', ha='center', fontsize=7, fontweight='bold')
ax1.set_xticks(x); ax1.set_xticklabels(mlabels)
ax1.set_ylim(0,1.1); ax1.legend(fontsize=8)
ax1.set_title('Metric Comparison — always show the baseline', fontsize=10)
ax1.spines[['top','right']].set_visible(False)

# ── Panels 2 & 3: Confusion matrices ──────────────────────────────
for ax_i,(preds,nm) in enumerate([(y_pred_lr,'Logistic Reg.'),(y_pred_rf,'Random Forest')],2):
    ax = fig.add_subplot(gs[0,ax_i])
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Stay','Leave'], yticklabels=['Stay','Leave'],
                linewidths=0.5, cbar=False)
    ax.set_title(f'{nm}\nConfusion Matrix', fontsize=9)
    ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')

# ── Panel 4: ROC curves ───────────────────────────────────────────
ax_roc = fig.add_subplot(gs[1,:2])
ax_roc.plot([0,1],[0,1],'k--',alpha=0.4,label='Random (AUC=0.50)')
for proba,nm in [(y_proba_lr,'Log.Reg.'),(y_proba_rf,'Rand.Forest')]:
    fpr,tpr,_ = roc_curve(y_test, proba)
    auc_v = roc_auc_score(y_test, proba)
    ax_roc.plot(fpr,tpr,linewidth=2,label=f'{nm} (AUC={auc_v:.3f})')
ax_roc.set_xlabel('False Positive Rate'); ax_roc.set_ylabel('True Positive Rate')
ax_roc.set_title('ROC Curves', fontsize=10)
ax_roc.legend(fontsize=8); ax_roc.spines[['top','right']].set_visible(False)

# ── Panel 5: Precision-Recall ─────────────────────────────────────
ax_pr = fig.add_subplot(gs[1,2:])
ax_pr.axhline(y_test.mean(), color='grey', ls='--', alpha=0.7,
               label=f'No-skill ({y_test.mean():.2f})')
for proba,nm in [(y_proba_lr,'Log.Reg.'),(y_proba_rf,'Rand.Forest')]:
    pc,rc,_ = precision_recall_curve(y_test, proba)
    ax_pr.plot(rc,pc,linewidth=2,label=nm)
ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')
ax_pr.set_title('Precision-Recall Curves', fontsize=10)
ax_pr.legend(fontsize=8); ax_pr.spines[['top','right']].set_visible(False)

plt.savefig('outputs/evaluation_dashboard.png', dpi=130, bbox_inches='tight')
plt.show()
print('💾 Saved: outputs/evaluation_dashboard.png')

---
# Section 4 — Error Analysis: Where Is the Model Failing?

> **The single most important insight in this notebook:**
> Hyperparameter tuning improves performance by 1–2 points.
> Finding that your model fails on a specific subgroup and fixing it
> can improve that subgroup's performance by 10–20 points.
> **Always do error analysis before tuning.**

### Three-Step Error Analysis

| Step | What you do | What you learn |
|------|------------|----------------|
| 1 — Quantify | Count errors by type | How many FP and FN? |
| 2 — Slice | Group by feature values | Which subgroups are worst? |
| 3 — Inspect | Look at individual failures | Why did the model get these wrong? |

In [ ]:
# ── STEP 1: QUANTIFY ERRORS ──────────────────────────────────────────────
test_df = X_test.copy()
test_df['actual']    = y_test.values
test_df['predicted'] = y_pred_rf
test_df['proba']     = y_proba_rf

# Label every prediction
test_df['error_type'] = 'Correct'
test_df.loc[(test_df['actual']==1)&(test_df['predicted']==0),'error_type'] = 'False Negative'
test_df.loc[(test_df['actual']==0)&(test_df['predicted']==1),'error_type'] = 'False Positive'

print('PREDICTION BREAKDOWN — Random Forest on test set')
print('='*52)
for etype, count in test_df['error_type'].value_counts().items():
    pct = count / len(test_df) * 100
    bar = '█' * int(pct / 2.5)
    print(f'  {etype:<22} {count:>5}  ({pct:>5.1f}%)  {bar}')

total_leavers = y_test.sum()
fn_count = (test_df['error_type']=='False Negative').sum()
fp_count = (test_df['error_type']=='False Positive').sum()
print(f'\nOf {total_leavers} actual leavers in test set:')
print(f'  Caught: {total_leavers - fn_count}  ({(1-fn_count/total_leavers)*100:.0f}%)')
print(f'  Missed: {fn_count}  ({fn_count/total_leavers*100:.0f}%)  ← false negatives')

In [ ]:
# ── STEP 2: SLICE-BASED ANALYSIS ─────────────────────────────────────────
# Find which subgroups the model handles poorly.
# This is where improvement opportunities live.

def slice_analysis(test_df, col):
    """Recall, precision, and miss rate for each subgroup of col."""
    print(f'\n{"─"*68}')
    print(f'  Error Analysis by: {col}')
    print(f'{"─"*68}')
    print(f'  {"Subgroup":<27} {"N":>5} {"Churn%":>7} {"Recall":>8} {"Miss%":>8}')

    for val, grp in test_df.groupby(col):
        n    = len(grp)
        cr   = grp['actual'].mean()
        rec  = recall_score(grp['actual'], grp['predicted'], zero_division=0)
        miss = ((grp['actual']==1)&(grp['predicted']==0)).mean()
        lbl  = f'{col}={val}'[:27]
        flag = '  ⚠️  LOW RECALL'  if rec < 0.50 else \
               '  ⚡ WATCH'        if rec < 0.65 else ''
        print(f'  {lbl:<27} {n:>5} {cr:>7.1%} {rec:>8.3f} {miss:>7.1%}{flag}')

# Run on the most informative features
for col in ['Department', 'OverTime', 'MaritalStatus', 'JobLevel']:
    slice_analysis(test_df, col)

In [ ]:
# ── STEP 2B: YEARS AT COMPANY SEGMENTATION ───────────────────────────────
# Tenure segments often reveal the model's biggest blind spots

test_df['tenure_segment'] = pd.cut(
    test_df['YearsAtCompany'],
    bins=[0, 2, 5, 10, 40],
    labels=['New (0-2y)', 'Early (2-5y)', 'Mid (5-10y)', 'Veteran (10y+)']
)

print('RECALL BY TENURE SEGMENT')
print('='*58)
seg_results = []
for seg, grp in test_df.groupby('tenure_segment', observed=True):
    n   = len(grp)
    rec = recall_score(grp['actual'], grp['predicted'], zero_division=0)
    miss= ((grp['actual']==1)&(grp['predicted']==0)).mean()
    seg_results.append({'seg':seg,'n':n,'recall':rec,'miss':miss})
    flag = '  ← BIGGEST GAP' if rec < 0.50 else ''
    print(f'  {str(seg):<20}: {n:>5} employees | recall={rec:.1%} | miss={miss:.1%}{flag}')

# Visualise tenure recall
sr_df = pd.DataFrame(seg_results)
overall_rec = recall_score(test_df['actual'], test_df['predicted'])

fig, ax = plt.subplots(figsize=(10, 4))
bar_colors = ['#D32F2F' if r < 0.50 else '#FFA000' if r < 0.65 else '#388E3C'
              for r in sr_df['recall']]
bars = ax.bar(sr_df['seg'].astype(str), sr_df['recall'], color=bar_colors, edgecolor='white')
ax.axhline(overall_rec, color='navy', linewidth=2, linestyle='--',
            label=f'Overall recall: {overall_rec:.1%}')
for bar, val in zip(bars, sr_df['recall']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{val:.1%}', ha='center', fontsize=11, fontweight='bold')
ax.set_title('Recall by Tenure Segment\n'
             'Red = model weakest here (<50%)', fontsize=11, fontweight='bold')
ax.set_ylabel('Recall (fraction of leavers caught)')
ax.set_ylim(0, 1.0); ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('outputs/tenure_recall.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── STEP 3: INSPECT INDIVIDUAL FALSE NEGATIVES ───────────────────────────
# Look at the cases where the model was MOST CONFIDENTLY wrong.
# Pattern matching here drives actionable improvements.

fn_df = test_df[test_df['error_type']=='False Negative'].copy()
fn_df = fn_df.sort_values('proba', ascending=True)   # most confident misses first

print(f'FALSE NEGATIVES — leavers the model missed ({len(fn_df)} total)')
print('Model was most certain these employees would STAY:')
print('='*65)

show = ['YearsAtCompany','MonthlyIncome','JobSatisfaction',
        'OverTime','Department','MaritalStatus','actual','proba']
avail = [c for c in show if c in fn_df.columns]
print(fn_df[avail].head(8).to_string(index=False))

print('\nPattern check in false negatives vs full test set:')
cols_check = ['YearsAtCompany','MonthlyIncome','JobSatisfaction']
for c in cols_check:
    fn_mean  = fn_df[c].mean()
    all_mean = test_df[c].mean()
    diff = fn_mean - all_mean
    print(f'  {c:<25} FN mean={fn_mean:>7.1f}  Overall={all_mean:>7.1f}  diff={diff:>+7.1f}')

if 'OverTime' in fn_df.columns:
    fn_ot  = (fn_df['OverTime']=='Yes').mean()
    all_ot = (test_df['OverTime']=='Yes').mean()
    print(f'  {"OverTime=Yes %":<25} FN={fn_ot:.1%}  Overall={all_ot:.1%}')

In [ ]:
# ── FORMALISE ERROR ANALYSIS FINDINGS ────────────────────────────────────
# This is the format used in professional ML project reviews.

ERROR_FINDINGS = [
    {
        'finding':  'New employees (0-2y tenure) have the lowest recall',
        'evidence': 'Recall drops to <50% for 0-2y tenure segment in slice analysis',
        'reason':   'New employees have limited history — fewer features carry signal',
        'fix':      'Add onboarding features: ramp-up score, first 90-day login rate',
        'priority': 'HIGH',
    },
    {
        'finding':  'OverTime=Yes employees are missed at 2x the average rate',
        'evidence': f'{(fn_df["OverTime"]=="Yes").mean():.0%} of FNs have OverTime=Yes '
                    f'vs {(test_df["OverTime"]=="Yes").mean():.0%} overall',
        'reason':   'Overtime is captured as binary — frequency and duration missing',
        'fix':      'Add avg_overtime_hours_per_week; add OverTime × LowSalary interaction',
        'priority': 'MEDIUM',
    },
    {
        'finding':  'HR department has worst recall of all departments',
        'evidence': 'Slice analysis: HR recall lowest across all three departments',
        'reason':   'HR is the smallest group (10%) — underrepresented in training',
        'fix':      'Oversample HR in training or collect 6 more months of HR data',
        'priority': 'LOW',
    },
]

print('📋 ERROR ANALYSIS FINDINGS')
print('='*60)
for i, f in enumerate(ERROR_FINDINGS, 1):
    print(f'\n  Finding {i}  [{f["priority"]}]')
    print(f'    What:     {f["finding"]}')
    print(f'    Evidence: {f["evidence"]}')
    print(f'    Why:      {f["reason"]}')
    print(f'    Fix:      {f["fix"]}')

---
# Section 5 — Actionable Improvement Suggestions

**The rule:** Every improvement suggestion must connect a **specific observed failure**
to a **specific proposed fix**. Vague suggestions are not improvements — they are wishes.

| ❌ Vague | ✅ Actionable |
|----------|---------------|
| 'I would get more data' | 'Collect 6 months of HR department data — currently only 10% of training examples' |
| 'I would try more features' | 'Add avg_overtime_hours_per_week — present in 58% of false negatives but not in current feature set' |
| 'I would tune the model' | 'Lower threshold from 0.50 → 0.40: recall improves from 69% to 81% at +12% false positives — HR confirmed this is acceptable' |

In [ ]:
# ── IMPROVEMENT 1: THRESHOLD ANALYSIS ────────────────────────────────────
# The fastest, highest-impact improvement: choose the right decision threshold.
# Always do this before retraining the model.

thresholds = np.arange(0.20, 0.81, 0.05)
thresh_results = []
for t in thresholds:
    p_t = (y_proba_rf >= t).astype(int)
    thresh_results.append({
        'threshold': t,
        'precision': precision_score(y_test, p_t, zero_division=0),
        'recall':    recall_score(y_test, p_t, zero_division=0),
        'f1':        f1_score(y_test, p_t, zero_division=0),
        'flagged':   int(p_t.sum()),
    })
thresh_df = pd.DataFrame(thresh_results)

print('Threshold Analysis: how changing the cut-off affects business outcomes')
print(f'{"Threshold":>10} {"Precision":>10} {"Recall":>8} {"F1":>6} {"Flagged":>9}')
print('─'*50)
for _,row in thresh_df[thresh_df['threshold'].isin([0.30,0.35,0.40,0.50,0.60,0.70])].iterrows():
    note = '← recommended' if abs(row['threshold']-0.40)<0.01 else \
           '← current'     if abs(row['threshold']-0.50)<0.01 else ''
    print(f"{row['threshold']:>10.2f} {row['precision']:>10.3f} "
          f"{row['recall']:>8.3f} {row['f1']:>6.3f} "
          f"{int(row['flagged']):>9}   {note}")

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Threshold Analysis — Business Impact of Changing the Decision Cut-off',
             fontsize=12, fontweight='bold')

axes[0].plot(thresh_df['threshold'], thresh_df['precision'],'b-o',ms=4,label='Precision')
axes[0].plot(thresh_df['threshold'], thresh_df['recall'],   'r-o',ms=4,label='Recall')
axes[0].plot(thresh_df['threshold'], thresh_df['f1'],       'g-o',ms=4,label='F1')
axes[0].axvline(0.50, color='grey',   ls='--', alpha=0.7, label='Current (0.50)')
axes[0].axvline(0.40, color='purple', ls='--', alpha=0.9, label='Proposed (0.40)')
axes[0].set_xlabel('Decision Threshold'); axes[0].set_ylabel('Score')
axes[0].set_title('Metric vs Threshold'); axes[0].legend(fontsize=8)
axes[0].spines[['top','right']].set_visible(False)

axes[1].bar(thresh_df['threshold'], thresh_df['flagged'], width=0.04,
             color=['#D32F2F' if abs(t-0.40)<0.01 else '#90CAF9'
                    for t in thresh_df['threshold']])
axes[1].set_xlabel('Threshold'); axes[1].set_ylabel('Employees Flagged for Outreach')
axes[1].set_title('Flagged Volume vs Threshold'); axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('outputs/threshold_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── PRIORITISED IMPROVEMENT ROADMAP ──────────────────────────────────────

ROADMAP = [
    {
        'priority': 1,
        'effort':   'Low — 1 day',
        'title':    'Lower decision threshold from 0.50 to 0.40',
        'evidence': 'Threshold analysis: recall improves from 69% to 81% at 0.40',
        'tradeoff': '+12% false positives — HR team confirmed this is acceptable',
        'expected': 'Catch ~35 additional leavers per quarter',
    },
    {
        'priority': 2,
        'effort':   'Medium — 3-5 days',
        'title':    'Add overtime intensity features',
        'evidence': f'{(fn_df["OverTime"]=="Yes").mean():.0%} of false negatives have OverTime=Yes '
                    f'vs {(test_df["OverTime"]=="Yes").mean():.0%} overall',
        'tradeoff': 'Requires extracting avg hours from payroll system',
        'expected': 'Estimated 5-8 F1 point improvement for OverTime subgroup',
    },
    {
        'priority': 3,
        'effort':   'High — 1-2 weeks',
        'title':    'Build separate model for new employees (tenure < 2y)',
        'evidence': 'Slice analysis: recall < 50% for 0-2y tenure segment',
        'tradeoff': 'Needs 6 more months of data; two models to maintain',
        'expected': 'Address ~30% of currently missed leavers in new-hire cohort',
    },
]

print('🗺️  PRIORITISED IMPROVEMENT ROADMAP')
print('='*65)
for item in ROADMAP:
    print(f"\n  Priority {item['priority']}  [{item['effort']}]")
    print(f"  What:       {item['title']}")
    print(f"  Evidence:   {item['evidence']}")
    print(f"  Trade-off:  {item['tradeoff']}")
    print(f"  Expected:   {item['expected']}")

---
# Section 6 — 5-Minute Demo Rehearsal

Five minutes is tight. The most common mistake: spending 4 minutes on methodology
and 30 seconds rushing through results.

**The audience remembers two things:**
1. Your headline result in business terms
2. Whether you seemed honest about limitations

They do not remember your preprocessing pipeline.

### Structure That Works Every Time

| Minute | Content | Show | Never |
|--------|---------|------|-------|
| 1 | Problem + Data | 3 sentences | List all features |
| 2 | Approach | 1 summary diagram | Walk through pipeline code |
| 3 | Results | 1 chart + headline in business terms | Every metric you computed |
| 4 | Where it fails + next steps | 2 findings + 2 fixes | Every weakness found |
| 5 | Live inference | 1 prediction | Multiple examples |

In [ ]:
# ── RESULTS CARD — the single visual for your demo ───────────────────────
def print_results_card(project, problem, data_desc, m_bl, m_rf):
    delta = m_rf['acc'] - m_bl['acc']
    print('╔' + '═'*64 + '╗')
    print(f'║  {"PROJECT RESULTS CARD":^60}  ║')
    print('╠' + '═'*64 + '╣')
    print(f'║  Project: {project:<53} ║')
    print(f'║  Problem: {problem:<53} ║')
    print(f'║  Data:    {data_desc:<53} ║')
    print('╠' + '═'*64 + '╣')
    print(f'║  {"Model":<28} {"Acc":>6} {"Prec":>6} {"Recall":>8} {"F1":>6} {"AUC":>7}  ║')
    print(f'║  {"─"*60}  ║')
    print(f'║  {"Baseline (always predict Stay)":<28} '
          f'{m_bl["acc"]:>6.3f} {"—":>6} {"0%":>8} {"—":>6} {"—":>7}  ║')
    print(f'║  {"Random Forest"::<28} '
          f'{m_rf["acc"]:>6.3f} {m_rf["prec"]:>6.3f} '
          f'{m_rf["rec"]:>7.1%} {m_rf["f1"]:>6.3f} {m_rf["auc"]:>7.3f}  ║')
    print('╠' + '═'*64 + '╣')
    print(f'║  Improvement over baseline:  {delta:+.3f} accuracy points{" "*27} ║')
    print(f'║  Biggest gap: new employee recall <50% — needs onboarding features{" "*6} ║')
    print(f'║  Quick win:   threshold 0.50→0.40 → recall +12pts at acceptable FP{" "*4} ║')
    print('╚' + '═'*64 + '╝')

print_results_card(
    project   = 'Employee Attrition Predictor',
    problem   = 'Flag at-risk employees before they resign',
    data_desc = '1,470 employees | 20 features | synthetic IBM HR data',
    m_bl      = m_bl,
    m_rf      = m_rf,
)

In [ ]:
# ── DEMO SCRIPT WITH WORD COUNT AND TIMING ───────────────────────────────
DEMO_SCRIPT = {
    'Minute 1 — Problem + Data': [
        'Employee attrition costs the organisation roughly 85,000 rupees per replacement hire.',
        'I built a model to identify which employees are at risk of leaving in the next quarter',
        'so HR can intervene before they resign.',
        'I used 1,470 employee records with 20 features including',
        'salary, tenure, job satisfaction, and overtime patterns.',
    ],
    'Minute 2 — Approach': [
        'I first built a baseline — always predict the employee will stay.',
        'That gives 80% accuracy but catches zero actual leavers.',
        'I then trained a Random Forest with class balancing to handle the 20% attrition rate.',
        'Five-fold stratified cross-validation on training data,',
        '20% held out as a truly unseen test set.',
    ],
    'Minute 3 — Results': [
        'The model correctly identifies 70% of employees who will leave,',
        'compared to zero for the baseline.',
        'ROC-AUC is 0.82 — strong discrimination ability.',
        'In business terms: of every 100 employees who will actually resign,',
        'we correctly flag 70 of them for HR intervention.',
    ],
    'Minute 4 — Honest Assessment + Next Steps': [
        'Honest finding: the model struggles with new employees under 2 years tenure.',
        'Recall drops below 50% for that cohort — we miss most of them.',
        'Root cause: new hires have limited history so fewer features carry signal.',
        'My top fix: lower the decision threshold from 0.50 to 0.40.',
        'This increases recall to 81% at an acceptable false-positive cost,',
        'which the HR team confirmed they can absorb.',
    ],
    'Minute 5 — Live Inference': [
        'Let me show one live prediction.',
        'Employee: 1 year tenure, OverTime Yes, low job satisfaction score.',
        'Predicted attrition probability — showing now.',
        'Questions?',
    ],
}

print('📋 DEMO SCRIPT — WORD COUNT AND TIMING')
print('='*52)
print(f'  Approximate speaking pace: 130 words per minute')
print()
total = 0
for section, lines in DEMO_SCRIPT.items():
    words = sum(len(l.split()) for l in lines)
    total += words
    mins  = words / 130
    status = '✅' if mins <= 1.1 else '⚠️  TOO LONG — CUT'
    print(f'  {section}')
    print(f'    {words} words  |  est. {mins:.1f} min  {status}')
    print()
print(f'  TOTAL: {total} words  |  est. {total/130:.1f} min')
print(f'  TARGET: ≤650 words  |  ≤5.0 min')
print()
if total > 650:
    print(f'  ⚠️  Over by {total-650} words. Cut ruthlessly — say less, more clearly.')
else:
    print('  ✅ Fits within 5 minutes. Read it aloud to verify.')

In [ ]:
# ── LIVE INFERENCE CELL ───────────────────────────────────────────────────
# This is the cell you run live during the final minute of your demo.
# Keep it simple: one employee, one prediction, one clear explanation.

new_employee = pd.DataFrame([{
    'Age':                  27,
    'MonthlyIncome':        28000,    # below-median income
    'YearsAtCompany':       1,        # new employee
    'YearsInCurrentRole':   1,
    'TotalWorkingYears':    2,
    'DistanceFromHome':     18,       # long commute
    'JobSatisfaction':      2,        # low satisfaction
    'WorkLifeBalance':      2,
    'PercentSalaryHike':    11,
    'TrainingTimesLastYear':0,
    'JobLevel':             1,
    'StockOptionLevel':     0,
    'NumCompaniesWorked':   3,
    'Department':           'Sales',
    'JobRole':              'Sales Exec',
    'MaritalStatus':        'Single',
    'OverTime':             'Yes',    # working overtime
    'BusinessTravel':       'Travel_Frequently',
    'EducationField':       'Marketing',
}])

proba = rf.predict_proba(new_employee)[0, 1]
pred  = rf.predict(new_employee)[0]

print('🔮  LIVE INFERENCE — Demo Employee Profile')
print('='*50)
print('  Profile: Sales Exec | Age 27 | 1yr tenure')
print('           OverTime=Yes | Satisfaction=2 | Low salary')
print()
print(f'  Attrition probability: {proba:.1%}')
print(f'  Decision (threshold=0.50): {"LIKELY TO LEAVE" if pred==1 else "LIKELY TO STAY"}')
print()
if proba > 0.50:
    print('  ✅ Recommendation: Flag for immediate HR retention outreach.')
elif proba > 0.40:
    print('  ⚡ At proposed threshold (0.40): would be flagged.')
    print('  Recommendation: Include in next retention review cycle.')
else:
    print('  Employee appears stable. Monitor in next quarterly review.')

---
# Section 7 — Peer Feedback Framework

**For the reviewer:** Specific feedback is kind. Vague feedback is useless.

| ❌ Vague | ✅ Specific |
|----------|-------------|
| 'Results were confusing' | 'I could not tell if 70% recall was good without seeing the baseline' |
| 'Looks good' | 'The threshold chart made the trade-off immediately clear' |
| 'You should improve the model' | 'The tenure finding was not connected to a specific fix. What would you actually do?' |

### The Three Questions That Always Surface Useful Feedback

1. **After Minute 1:** Can you state the problem in your own words? *(If not, communication failed.)*
2. **After Minute 4:** What was the strongest evidence for any finding? *(If unclear, error analysis was too vague.)*
3. **At the end:** What one question do you still have? *(That question is the presenter's most important feedback.)*

In [ ]:
# ── STRUCTURED PEER FEEDBACK FORM ────────────────────────────────────────
# Fill this in for your partner's demo.
# Rate each section 1-5. Write one specific comment per section.

FEEDBACK_CRITERIA = {
    'Problem Clarity (Minute 1)': {
        'question': 'After hearing Minute 1, could you state the problem in your own words?',
        'good':     'Yes — in one clear sentence, without needing to ask follow-up questions.',
    },
    'Honest Metrics (Minute 3)': {
        'question': 'Did the presenter compare to a baseline AND translate scores to business terms?',
        'good':     'Baseline shown. Metric matches business problem. Numbers stated as employees not percentages.',
    },
    'Error Analysis Depth (Minute 4)': {
        'question': 'Were failures described specifically, with evidence, sliced by subgroup?',
        'good':     'Not "the model sometimes fails" but "recall drops to X% for Y subgroup".',
    },
    'Improvement Suggestions (Minute 4)': {
        'question': 'Were next steps specific and connected to observed failures?',
        'good':     'Not "try more features" but "add X feature because it appears in Y% of false negatives".',
    },
    'Live Demo (Minute 5)': {
        'question': 'Did the demo run cleanly? Was the output explained in business terms?',
        'good':     'Demo ran without errors. Prediction was explained, not just shown.',
    },
    'Time Management': {
        'question': 'Did the demo finish in 5 minutes without rushing at the end?',
        'good':     'Finished naturally. Time for 1-2 questions.',
    },
}

print('📝 PEER FEEDBACK FORM')
print('='*60)
print(f'  Presenter: _________________________  Date: _____________\n')
for section, details in FEEDBACK_CRITERIA.items():
    print(f'  ─────────────────────────────────────────────────')
    print(f'  {section}')
    print(f'  Q: {details["question"]}')
    print(f'  What good looks like: {details["good"]}')
    print(f'  Rating (1-5): ___   Comment: __________________________________')
    print()
print('  ─────────────────────────────────────────────────')
print('  Top strength:               _______________________________________')
print('  Most important improvement: _______________________________________')
print('  Question still unanswered:  _______________________________________')

---
# 🏋️ Exercises

Complete all five. These build the habits that make good analysts great ones.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# EXERCISE 1 — Interpret These Metrics Honestly
# ══════════════════════════════════════════════════════════════════
# A colleague shows you these fraud detection model results:
#   Accuracy:  98.7%  |  Precision: 0.41  |  Recall: 0.23  |  AUC: 0.79
#   Majority-class baseline accuracy: 98.3%
#
# Answer all of the following in comments:
# (a) Is 98.7% accuracy impressive? Why or why not?
# (b) What does recall of 23% mean in plain English for fraud detection?
# (c) Which single metric should be prioritised for this use case, and why?
# (d) Write the 2-sentence honest interpretation you would say in a demo.

# YOUR ANSWERS:
# (a) ...
# (b) ...
# (c) ...
# (d) ...
print('Exercise 1: Write your answers as comments above.')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# EXERCISE 2 — Run Your Own Slice Analysis
# ══════════════════════════════════════════════════════════════════
# Using the test_df already built in Section 4:
#
# (a) Run slice_analysis() on 'JobLevel'
# (b) Create a new column: high_distance = 1 if DistanceFromHome > 15
#     Run slice_analysis() on 'high_distance'
#
# For each analysis answer:
#   - Which subgroup has the lowest recall?
#   - Is this a fixable problem or a data limitation?

print('Exercise 2: Run slice analysis on JobLevel and high_distance')
# (a)
# slice_analysis(test_df, 'JobLevel')

# (b)
# test_df['high_distance'] = (test_df['DistanceFromHome'] > 15).astype(int)
# slice_analysis(test_df, 'high_distance')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# EXERCISE 3 — Make These Suggestions Actionable
# ══════════════════════════════════════════════════════════════════
# Convert each vague statement into a specific, evidence-backed suggestion.
# Use the pattern: [specific change] because [evidence from this project].

vague = [
    'I would collect more data for underperforming groups.',
    'I would try adding more features to improve recall.',
    'I would consider adjusting the model hyperparameters.',
]

print('Vague → Actionable Suggestions')
print('='*55)
for i, v in enumerate(vague, 1):
    print(f'\nVague {i}:     {v}')
    print(f'Actionable {i}: [YOUR ANSWER — use evidence from Section 4 error analysis]')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# EXERCISE 4 — Write and Time Your Own Demo Script
# ══════════════════════════════════════════════════════════════════
# Write your own 5-minute demo script for this attrition project.
# Fill in every minute. Count the words. Cut if over 650.

MY_SCRIPT = {
    'Minute 1': ['[YOUR SENTENCE 1]', '[YOUR SENTENCE 2]', '[YOUR SENTENCE 3]'],
    'Minute 2': ['[YOUR SENTENCE 1]', '[YOUR SENTENCE 2]'],
    'Minute 3': ['[HEADLINE IN BUSINESS TERMS]', '[BASELINE COMPARISON]', '[TRANSLATION]'],
    'Minute 4': ['[SPECIFIC FAILURE WITH EVIDENCE]', '[SPECIFIC FIX]', '[SECOND FIX]'],
    'Minute 5': ['[WHAT YOU WILL DEMO]', '[HOW YOU WILL EXPLAIN THE OUTPUT]'],
}

total_words = sum(len(s.split()) for lines in MY_SCRIPT.values() for s in lines)
print(f'Your script: {total_words} words | est. {total_words/130:.1f} min')
print(f'Target:      <650 words  | <5.0 min')
if total_words > 650:
    print(f'⚠️  Over by {total_words-650} words. Cut ruthlessly.')
else:
    print('✅ Fits. Now read it aloud and time yourself.')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# EXERCISE 5 — Apply the Full Review Cycle to Your Own Project
# ══════════════════════════════════════════════════════════════════
# Take the project you built in the baseline session and complete:
#
# Step 1: Run the structure checklist on your notebook
# Step 2: Run honest_metrics() on your test results
# Step 3: Run slice_analysis() on 2-3 subgroups
# Step 4: Write 3 improvement suggestions using the roadmap format
# Step 5: Write your 5-minute demo script, count words, read aloud
#
# Deliverable: a results card + a written demo script for YOUR project.

print('Exercise 5: Full review cycle for your own project.')
print()
checklist = [
    'Run notebook structure_checklist() — fix any issues found',
    'Run honest_metrics() on your test results — write the business translation',
    'Run slice_analysis() on at least 2 meaningful subgroups',
    'Write 3 improvement suggestions using ROADMAP format with evidence',
    'Write demo script — count words — read aloud — time yourself',
]
for i, step in enumerate(checklist, 1):
    print(f'  {i}. [ ] {step}')

---
## 📋 Session Summary

| Skill | The Core Habit |
|-------|----------------|
| Code review | Restart kernel → Run All → same results? Fix it before sharing. |
| Honest metrics | Always show the baseline. Translate to business units. Never cherry-pick. |
| Error analysis | Slice by subgroup. Inspect individual failures. Ask: why? Is this fixable? |
| Improvements | Connect every suggestion to a specific observed failure. Evidence first. |
| 5-min demo | Problem → Approach → Results → Failures → Live demo. Time yourself. |
| Peer feedback | Specific over vague. One strength. One improvement. One unanswered question. |

### The Single Most Important Habit

> **Acknowledge what your model gets wrong, explain why, and propose a specific fix.**
> Anyone can report their best number.
> Only a good analyst can explain their worst number.

---

*Notebook prepared for the AIM Program — IIT Patna × Masai School*
*Mini-Project: Review & Present*